# Pure exogenous HDM error correction
This notebook implements a **strictly causal, non-autoregressive, direct multi-horizon model** for HDM error correction.

The design choice is deliberate:

- **No autoregression on the target**.
- Prediction uses only **past and current exogenous forcings**.
- Each horizon \(h\) is modeled separately:

\[
\hat{\varepsilon}(t+h) = f_h\big(X(t), X(t-1), \dots, X(t-L)\big)
\]

This is a **direct transfer-function / FIR-style regression** with exogenous lagged inputs only.

Why not use `ForecasterDirect` directly for training here?
Because in `skforecast`, the forecasters are built around `lags` and/or `window_features` of the target series. For a **strict no-AR specification**, the safest implementation is:

1. use **`skforecast.model_selection.TimeSeriesFold`** for honest time-series CV,
2. build the lagged exogenous design matrix explicitly,
3. fit one **Ridge** model per horizon.

In [ ]:
# Cell 1 — Imports and global config
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from skforecast.model_selection import TimeSeriesFold

RANDOM_STATE = 42

# -----------------------------------------------------------------------------
# USER CONFIG
# -----------------------------------------------------------------------------
DATA_DIR = Path('/Users/nicolocaron/Desktop/MASTER PROJECT/data/per_station')

STATIONS = {
    'Sonderborg': {'id': 26473, 'lat': 54.91, 'lon':  9.79},
    'Assens':     {'id': 28366, 'lat': 55.27, 'lon':  9.89},
    'Bagenkop':   {'id': 28548, 'lat': 54.75, 'lon': 10.67},
    'Kobenhavn':  {'id': 30336, 'lat': 55.69, 'lon': 12.60},
    'Koge':       {'id': 30478, 'lat': 55.45, 'lon': 12.18},
    'Gedser':     {'id': 31616, 'lat': 54.57, 'lon': 11.93},
}

# Core atmospheric predictors
BASE_EXOG = ['SLP', 't2m', 'u10', 'v10']

# Optional tide predictors, used only if present in the parquet
OPTIONAL_EXOG = ['tide_M2', 'tide_K1']

# Maximum lag in hours
MAX_LAG = 96

# Dense lag set. If you want a sparser FIR, replace with:
# EXOG_LAGS = [0, 1, 2, 3, 6, 12, 24, 48, 72, 96]
EXOG_LAGS = list(range(0, MAX_LAG + 1))

# Forecast horizons in hours
HORIZONS = [1, 3, 6, 12, 24]

# Candidate alphas for Ridge tuning
ALPHAS = [0.1, 1.0, 10.0, 100.0, 1000.0]

# Initial train fraction for outer evaluation
TRAIN_FRAC = 2/3

print('Config loaded.')
print(f'MAX_LAG = {MAX_LAG} h')
print(f'Number of lagged exogenous columns with 4 base variables = {len(BASE_EXOG) * len(EXOG_LAGS)}')

## Important methodological choices

This notebook fixes the main conceptual issues:

1. **No target lags are used anywhere.**
2. **Bias removal is train-only.**  
   The station mean error is estimated on the training block only, then subtracted from both train and test targets.
3. **Scaling is fold-safe.**  
   `StandardScaler` is inside a `Pipeline`, so it is fit only on the training fold.
4. **Direct strategy is explicit.**  
   One Ridge model is trained for each horizon.
5. **Lagged exogenous matrix is built explicitly.**  
   You fully control the causal structure.

In [ ]:
# Cell 2 — Data loading
def load_station_dataframe(
    station_name: str,
    data_dir: Path = DATA_DIR,
    model_col: str = 'forcoast_p82_m',
    obs_col: str = 'tg_obs_m',
    freq: str = 'h',
    interp_limit: int = 6
) -> pd.DataFrame:
    """
    Load one station parquet and return a clean hourly DataFrame.

    Required columns:
        - model_col
        - obs_col
        - exogenous variables listed in BASE_EXOG and OPTIONAL_EXOG if present

    Returned columns:
        - model_col
        - obs_col
        - error_raw = model - obs
        - selected exogenous predictors
    """
    meta = STATIONS[station_name]
    path = data_dir / f"station_{meta['id']}_{station_name}.parquet"
    df = (
        pd.read_parquet(path)
        .assign(time=lambda x: pd.to_datetime(x['time']))
        .sort_values('time')
        .drop_duplicates('time')
        .set_index('time')
    )

    full_idx = pd.date_range(df.index.min(), df.index.max(), freq=freq)
    df = df.reindex(full_idx)
    df.index.freq = freq

    df['error_raw'] = df[model_col] - df[obs_col]

    # Keep only available exogenous variables
    exog_cols = [c for c in BASE_EXOG + OPTIONAL_EXOG if c in df.columns]

    keep_cols = [model_col, obs_col, 'error_raw'] + exog_cols
    df = df[keep_cols].copy()

    # Short interpolation only
    for c in keep_cols:
        df[c] = df[c].interpolate(method='time', limit=interp_limit)

    return df


# Quick example
df_kbh = load_station_dataframe('Kobenhavn')
print(df_kbh.head())
print('\nColumns:', list(df_kbh.columns))

In [ ]:
# Cell 3 — Build lagged exogenous matrix
def build_lagged_exog(
    df: pd.DataFrame,
    exog_cols: list[str],
    lags: list[int]
) -> pd.DataFrame:
    """
    Create a strictly causal lagged exogenous matrix.
    At row t, column SLP_lag6 contains SLP(t-6h).
    """
    frames = []
    for col in exog_cols:
        for lag in lags:
            s = df[col].shift(lag)
            s.name = f'{col}_lag{lag}'
            frames.append(s)
    X = pd.concat(frames, axis=1)
    return X


def make_direct_dataset(
    df: pd.DataFrame,
    exog_cols: list[str],
    lags: list[int],
    horizon: int,
    target_col: str = 'error_raw'
) -> tuple[pd.DataFrame, pd.Series]:
    """
    Build direct supervised dataset for horizon h:
        X_t   = exogenous lags available up to time t
        y_t   = error at time t+h
    """
    X = build_lagged_exog(df, exog_cols=exog_cols, lags=lags)
    y = df[target_col].shift(-horizon).rename(f'{target_col}_plus_{horizon}h')

    out = pd.concat([X, y], axis=1).dropna()
    y_out = out[y.name].copy()
    X_out = out.drop(columns=[y.name]).copy()

    return X_out, y_out


exog_cols_example = [c for c in BASE_EXOG + OPTIONAL_EXOG if c in df_kbh.columns]
X12, y12 = make_direct_dataset(df_kbh, exog_cols_example, EXOG_LAGS, horizon=12)

print('Exogenous variables used:', exog_cols_example)
print('X shape:', X12.shape)
print('y shape:', y12.shape)
print('First columns:', list(X12.columns[:8]))

## Fold-safe bias correction

Vahid-style target decomposition is:

\[
E(t) = \varepsilon(t) + \mathrm{RefBias}
\]

and the dynamic training target is the demeaned residual, station by station.  
Here I enforce the statistically honest version:

- estimate the mean bias **only on train**,
- subtract that bias from both train and test target in the fold,
- keep that bias and reapply it consistently at final fit time.

In [ ]:
# Cell 4 — Time-series split helpers
def fold_dataframe_from_skforecast(cv, index_like):
    """
    Convert TimeSeriesFold output to a plain pandas DataFrame.
    Compatible with current skforecast output style.
    """
    folds = cv.split(X=index_like, as_pandas=True)
    if isinstance(folds, pd.DataFrame):
        return folds.copy()
    # Fallback in case a list-like object is returned
    return pd.DataFrame(folds)


def get_train_test_slices_from_foldrow(row):
    """
    Robust extraction of train/test slice bounds from a fold row produced by TimeSeriesFold.
    """
    # Common columns in current docs / implementations can vary slightly.
    # We therefore search by semantic names.
    cols = {str(c).lower(): c for c in row.index}

    def pick(*candidates):
        for cand in candidates:
            for low, orig in cols.items():
                if cand == low:
                    return row[orig]
        return None

    train_start = pick('train_start', 'train_start_idx')
    train_end   = pick('train_end', 'train_end_idx')
    test_start  = pick('test_start', 'test_start_idx')
    test_end    = pick('test_end', 'test_end_idx')

    if train_start is None or train_end is None or test_start is None or test_end is None:
        raise ValueError(
            f'Unexpected fold format. Available columns: {list(row.index)}'
        )
    return int(train_start), int(train_end), int(test_start), int(test_end)

In [ ]:
# Cell 5 — Inner CV for alpha tuning (pure exogenous direct regression)
def evaluate_alpha_inner_cv(
    X: pd.DataFrame,
    y: pd.Series,
    alpha: float,
    n_initial_train: int,
    horizon: int
) -> float:
    """
    Evaluate one alpha by time-series CV.
    Returns mean validation RMSE in meters.
    """
    cv = TimeSeriesFold(
        steps=horizon,
        initial_train_size=n_initial_train,
        refit=False,
        fixed_train_size=True,
        allow_incomplete_fold=False,
        verbose=False
    )
    folds = fold_dataframe_from_skforecast(cv, X)

    rmses = []

    for _, row in folds.iterrows():
        tr0, tr1, te0, te1 = get_train_test_slices_from_foldrow(row)

        X_train = X.iloc[tr0:tr1+1].copy()
        y_train = y.iloc[tr0:tr1+1].copy()
        X_valid = X.iloc[te0:te1+1].copy()
        y_valid = y.iloc[te0:te1+1].copy()

        # Train-only bias removal
        bias_train = y_train.mean()
        y_train_dyn = y_train - bias_train
        y_valid_dyn = y_valid - bias_train

        model = Pipeline([
            ('scaler', StandardScaler()),
            ('ridge', Ridge(alpha=alpha))
        ])
        model.fit(X_train, y_train_dyn)
        pred = model.predict(X_valid)

        rmse = np.sqrt(mean_squared_error(y_valid_dyn, pred))
        rmses.append(rmse)

    return float(np.mean(rmses))


def tune_alpha(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    horizon: int,
    alphas: list[float] = ALPHAS
) -> tuple[float, pd.DataFrame]:
    """
    Time-series tuning of alpha on the training block only.
    """
    n_initial = max(int(len(X_train) * 0.6), 500)

    rows = []
    for alpha in alphas:
        mean_rmse = evaluate_alpha_inner_cv(
            X=X_train,
            y=y_train,
            alpha=alpha,
            n_initial_train=n_initial,
            horizon=horizon
        )
        rows.append({'alpha': alpha, 'cv_rmse_m': mean_rmse})

    results = pd.DataFrame(rows).sort_values('cv_rmse_m').reset_index(drop=True)
    best_alpha = float(results.loc[0, 'alpha'])
    return best_alpha, results

In [ ]:
# Cell 6 — Train and evaluate one station / one horizon
def fit_direct_exog_station_horizon(
    station_name: str,
    horizon: int,
    data_dir: Path = DATA_DIR,
    lags: list[int] = EXOG_LAGS,
    train_frac: float = TRAIN_FRAC,
    alphas: list[float] = ALPHAS
) -> dict:
    """
    Strictly pure exogenous direct model for one station and one horizon.
    """
    df = load_station_dataframe(station_name, data_dir=data_dir)
    exog_cols = [c for c in BASE_EXOG + OPTIONAL_EXOG if c in df.columns]

    X, y = make_direct_dataset(
        df=df,
        exog_cols=exog_cols,
        lags=lags,
        horizon=horizon,
        target_col='error_raw'
    )

    n_total = len(X)
    n_train = int(n_total * train_frac)

    X_train, X_test = X.iloc[:n_train].copy(), X.iloc[n_train:].copy()
    y_train, y_test = y.iloc[:n_train].copy(), y.iloc[n_train:].copy()

    best_alpha, tuning_table = tune_alpha(
        X_train=X_train,
        y_train=y_train,
        horizon=horizon,
        alphas=alphas
    )

    # Final train-only bias removal
    bias_train = y_train.mean()
    y_train_dyn = y_train - bias_train
    y_test_dyn  = y_test - bias_train

    model = Pipeline([
        ('scaler', StandardScaler()),
        ('ridge', Ridge(alpha=best_alpha))
    ])
    model.fit(X_train, y_train_dyn)
    pred_test_dyn = pd.Series(model.predict(X_test), index=y_test.index, name='pred')

    metrics = {
        'station': station_name,
        'horizon_h': horizon,
        'best_alpha': best_alpha,
        'bias_train_m': float(bias_train),
        'r2_test': float(r2_score(y_test_dyn, pred_test_dyn)),
        'rmse_test_m': float(np.sqrt(mean_squared_error(y_test_dyn, pred_test_dyn))),
        'mae_test_m': float(mean_absolute_error(y_test_dyn, pred_test_dyn)),
        'n_train': int(len(X_train)),
        'n_test': int(len(X_test)),
        'n_features': int(X.shape[1]),
        'exog_cols': exog_cols
    }

    out = {
        'metrics': metrics,
        'model': model,
        'tuning_table': tuning_table,
        'X_train': X_train,
        'X_test': X_test,
        'y_train_raw': y_train,
        'y_test_raw': y_test,
        'y_train_dyn': y_train_dyn,
        'y_test_dyn': y_test_dyn,
        'pred_test_dyn': pred_test_dyn
    }
    return out


# Example
res_example = fit_direct_exog_station_horizon('Kobenhavn', horizon=12)
res_example['metrics']

In [ ]:
# Cell 7 — Run all stations and horizons
ALL_RESULTS = {}
rows = []

for station_name in STATIONS.keys():
    ALL_RESULTS[station_name] = {}
    for h in HORIZONS:
        result = fit_direct_exog_station_horizon(
            station_name=station_name,
            horizon=h
        )
        ALL_RESULTS[station_name][h] = result
        rows.append(result['metrics'])
        m = result['metrics']
        print(
            f"{station_name:11s}  h={h:2d}h  "
            f"R²={m['r2_test']:+.3f}  "
            f"RMSE={100*m['rmse_test_m']:.2f} cm  "
            f"alpha={m['best_alpha']}"
        )

SUMMARY = pd.DataFrame(rows)
SUMMARY

In [ ]:
# Cell 8 — Summary tables
df_r2 = SUMMARY.pivot(index='station', columns='horizon_h', values='r2_test').sort_index()
df_rmse_cm = 100 * SUMMARY.pivot(index='station', columns='horizon_h', values='rmse_test_m').sort_index()

print('R² test')
display(df_r2.round(3))

print('RMSE test [cm]')
display(df_rmse_cm.round(2))

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(df_r2.values, aspect='auto', cmap='RdYlGn', vmin=-0.5, vmax=1.0)
ax.set_xticks(range(len(df_r2.columns)))
ax.set_xticklabels([f'h={c}h' for c in df_r2.columns])
ax.set_yticks(range(len(df_r2.index)))
ax.set_yticklabels(df_r2.index)
for i in range(df_r2.shape[0]):
    for j in range(df_r2.shape[1]):
        ax.text(j, i, f'{df_r2.values[i, j]:.2f}', ha='center', va='center', fontsize=9)
ax.set_title('Pure exogenous direct Ridge — R² test')
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## Interpreting coefficients as an impulse response

Because the model is linear and uses only lagged exogenous inputs, the fitted coefficient vector for a given horizon is an empirical **finite impulse response**.

If \(\beta_{\text{SLP},6}^{(12)}\) is large in magnitude, it means:

- SLP from 6 hours ago matters for the 12-hour-ahead HDM error,
- the model has learned a delayed atmospheric influence,
- the sign tells you whether that forcing tends to increase or decrease the residual.

In [ ]:
# Cell 9 — Plot coefficients / impulse response by variable
def plot_irf(result_dict, station_name: str, horizon: int):
    model = result_dict['model']
    X_train = result_dict['X_train']
    coefs = model.named_steps['ridge'].coef_

    coef_s = pd.Series(coefs, index=X_train.columns)

    exog_used = result_dict['metrics']['exog_cols']
    fig, axes = plt.subplots(1, len(exog_used), figsize=(4.5 * len(exog_used), 4), sharey=True)
    if len(exog_used) == 1:
        axes = [axes]

    for ax, var in zip(axes, exog_used):
        sub = coef_s[[c for c in coef_s.index if c.startswith(f'{var}_lag')]].copy()
        lags = [int(c.split('_lag')[-1]) for c in sub.index]
        sub.index = lags
        sub = sub.sort_index()

        ax.plot(sub.index, sub.values, lw=2)
        ax.axhline(0, color='k', ls='--', lw=0.8)
        peak_lag = int(sub.abs().idxmax())
        ax.axvline(peak_lag, color='tab:red', ls=':', lw=1)
        ax.set_title(f'{var}  | peak lag={peak_lag}h')
        ax.set_xlabel('Lag [h]')
        ax.set_ylabel('Coefficient')

    m = result_dict['metrics']
    fig.suptitle(
        f'{station_name} | h={horizon}h | pure exogenous direct Ridge\n'
        f'R²={m["r2_test"]:+.3f} | RMSE={100*m["rmse_test_m"]:.2f} cm | alpha={m["best_alpha"]}'
    )
    plt.tight_layout()
    plt.show()


plot_irf(ALL_RESULTS['Kobenhavn'][12], 'Kobenhavn', 12)

In [ ]:
# Cell 10 — Event plot
def plot_event_window(
    station_name: str,
    horizon: int,
    center_time: str,
    half_window_hours: int = 120
):
    meta = STATIONS[station_name]
    df = load_station_dataframe(station_name)
    result = ALL_RESULTS[station_name][horizon]

    t0 = pd.Timestamp(center_time)
    t1 = t0 - pd.Timedelta(hours=half_window_hours)
    t2 = t0 + pd.Timedelta(hours=half_window_hours)

    pred = result['pred_test_dyn'].loc[t1:t2]
    y_test_raw = result['y_test_raw'].loc[t1:t2]
    bias_train = result['metrics']['bias_train_m']

    # Convert dynamic prediction back to raw error scale for visualization
    pred_raw = pred + bias_train

    ev = df.loc[t1:t2].copy()
    common = ev.index.intersection(pred_raw.index)

    fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

    axes[0].plot(ev.index, ev['tg_obs_m'], label='TG obs', color='black', lw=1.5)
    axes[0].plot(ev.index, ev['forcoast_p82_m'], label='HDM raw', color='tab:blue', lw=1.2, alpha=0.8)

    if len(common) > 0:
        hdm_corr = ev.loc[common, 'forcoast_p82_m'] - pred_raw.loc[common]
        axes[0].plot(common, hdm_corr, label=f'HDM corrected h={horizon}h', color='tab:green', lw=1.4)

    axes[0].set_ylabel('Sea level [m]')
    axes[0].legend()
    axes[0].set_title(f'{station_name} | event centered at {center_time} | pure exogenous model')

    axes[1].plot(ev.index, ev['error_raw'], label='Observed raw error', color='black', lw=1.5)
    if len(common) > 0:
        axes[1].plot(common, pred_raw.loc[common], label='Predicted raw error', color='tab:red', lw=1.3)
    axes[1].axhline(0, color='gray', ls='--', lw=0.8)
    axes[1].set_ylabel('HDM error [m]')
    axes[1].set_xlabel('Time')
    axes[1].legend()

    plt.tight_layout()
    plt.show()


# Example:
# plot_event_window('Kobenhavn', 12, center_time='2018-09-20', half_window_hours=120)

## What to do next
This notebook is the correct linear baseline for your scientific question.

After this, the logical sequence is:

1. compare **without tides** vs **with tides as predictors**,
2. test **sparser lag sets** to reduce collinearity,
3. add **station-upstream predictors** only if physically justified,
4. move to the causal CNN/TCN only after this linear baseline is stable.